# M7 Wiring Probe (launcher)

Colab launcher for `validation/m7_wiring_probe.py` (M7.2 sim-GT isolated arm probe).
Notebook responsibilities: env setup, VLA server health, run one script, print artifacts.

## 1 — Secrets, clone

In [ ]:
import getpass
github_token = getpass.getpass('GitHub PAT (repo scope): ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    print('check')
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt
%pip install -e .

## 3 — SmolVLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', 'services/requirements-vla-smolvla-linux.txt')

## 4 — `.env` + start VLA server + health

In [ ]:
import os, subprocess, time, urllib.request

with open('.env', 'w', encoding='utf-8') as f:
    f.write('VLA_POLICY=smolvla\nVLA_PORT=8765\nMUJOCO_GL=egl\n')

VLA_LOG = '/tmp/m7_wiring_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, 'VLA_POLICY': 'smolvla', 'VLA_PORT': '8765'},
)

health_ok = False
for elapsed in range(0, 180, 5):
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy; see {VLA_LOG}')

## 5 — Run M7.2 probe script

In [ ]:
import glob, json, os, subprocess, sys

env = {**os.environ,
       'MUJOCO_GL': 'egl',
       'VLA_POLICY': 'smolvla',
       'VLA_CHUNK_STEPS': '1',
       'PERCEPTION_BACKEND': 'sim_gt'}

result = subprocess.run(
    [sys.executable, 'validation/m7_wiring_probe.py', '--gate', 'm7.2'],
    cwd='/content/Longor',
    env=env,
    capture_output=True,
    text=True,
)
print(result.stdout[-4000:])
if result.stderr:
    print(result.stderr[-4000:])
print('exit_code=', result.returncode)

summaries = sorted(glob.glob('artifacts/colab/m7/*/summary.json'))
if summaries:
    latest = summaries[-1]
    print('summary.json:', latest)
    print(json.dumps(json.load(open(latest, encoding='utf-8')), indent=2)[-2000:])
else:
    print('No summary.json found under artifacts/colab/m7/')

if result.returncode != 0:
    raise SystemExit(result.returncode)